# 第49课 · 把 40 个相关数字拧成 13 个独立系数——DCT-II（离散余弦变换）纯 NumPy 手写

**目标**：从零实现 DCT-II，理解它如何把相关的 Mel 特征向量压缩成近似独立的倒谱系数。

🔗 Aurora 连接：`aurora.audio.mfcc.dct_ii()` 是本节的标准答案，最终 MFCC 流水线直接调用它。

← **上一课**　[L48 · 时频图解](L48_visual_stft.ipynb)

> 上节课学习了 **时频图解**：线性谱 / Mel 谱 / 对数 Mel 谱三者视觉对比。  
> 本课将探讨 **DCT-II 离散余弦变换**。

## 本课剧情：为什么 MFCC 要在 log-Mel 之后再做一次变换？

想象你录了一段对话，生成了 80 维 log-Mel 特征。这 80 个数字听起来很完整——但有个问题：相邻的 Mel 通道之间有约 50% 的重叠。这意味着第 10 和第 11 个数字几乎在说同一件事（相关系数可达 0.9+）。

把高度相关的特征送进神经网络，等于告诉它"两件事情很重要"——但其实是同一件事。浪费了容量，也让优化更难。

**DCT-II（离散余弦变换）的解法**：把 N 个相关数字"旋转"到另一个坐标系，使得大部分信息集中在前几个维度，剩余维度接近零。就像 PCA——只不过 DCT 的基是固定的余弦函数，不需要数据驱动。

**正交归一化 DCT-II 公式**：

$$X[k] = w(k)\sum_{n=0}^{N-1} x[n]\cos\!\left(\frac{\pi k(2n+1)}{2N}\right)$$

其中 $w(0)=\frac{1}{\sqrt{N}}$，$w(k>0)=\sqrt{\frac{2}{N}}$。

**为什么取前 13 维（MFCC-13）？**  
DCT 把能量集中到低频系数——前 13 维代表了缓慢变化的声道形状（音素），而高维系数对应快速振动（不稳定噪声）。截断 = 降维 + 去噪。

本节任务：实现 `dct_ii(x)` — 纯 NumPy，与 `scipy.fft.dct(norm='ortho')` 误差 < 1e-10。

## 先想一想：为什么不能直接把 40 维 Mel 喂给模型？

在往下看 DCT 公式之前，先自己想 30 秒：

1. 相邻的 Mel 滤波器有约 50% 重叠，那第 10 维和第 11 维的数值，是"两件独立的事"，还是"同一件事说了两遍"？
2. 如果这 40 个数字里塞满了重复信息，直接喂给模型，模型还得自己花力气发现"这俩其实一样"——这是帮忙还是添乱？

想通这两点，你就抓住了本课的动机：**DCT 不是数学炫技，而是把 40 个高度相关的数字去相关（decorrelation）、压缩成十几个近乎独立的系数**。

## 🤔 为什么工程师要发明它？(Why did engineers invent this?)
- **不用它会怎样？** 直接用 40 维 log-Mel，相邻维度相关系数高达 0.9+，等于反复告诉模型同一件事，浪费容量、拖慢优化；GMM / 欧氏距离这类"假设特征独立"的模型甚至会失效。
- **它解决了什么真实问题？** 用一组**固定的**余弦基，把相关信息旋转、集中到前几维（像不用训练的 PCA），后面维度接近 0，可以安全截断 → 既降维又去噪。
- **后面哪里还会再用到？** L50 MFCC 流水线的最后一步就是它；截到前 13 维即经典的 MFCC-13。

In [ ]:
import numpy as np

## 0. 先看 FFT vs DCT-II：它们不是同一个东西

| | FFT / DFT | DCT-II |
|---|---|---|
| 基函数 | 复指数（complex exponential） | 余弦 |
| 边界假设 | 圆周 / 周期延拓 | 偶对称延拓 |
| 输出 | 复数频谱 | 实数倒谱系数 |
| 典型用途 | 频谱分析 | 去相关 / 压缩 |
| Aurora | `aurora.audio.transforms.fft` | `aurora.audio.mfcc.dct_ii` |

DCT-II 放在 log-Mel 后面，是因为相邻 Mel 通道高度相关；
DCT 把重复信息旋转到更少的坐标里。
如果忘了为什么要做这一步，回看 L48 末尾的动机附录。

## ⚠️ 常见误解 (Common Pitfall)
> 不要把 DCT 理解成"另一种 FFT"。它其实是一次**固定基的去相关旋转**：FFT 用复指数基做频谱分析、输出复数、假设信号周期延拓；DCT-II 用纯余弦基、输出实数、假设偶对称延拓，目标是把相关能量压到低阶系数上。上表已逐行对照两者——记住"同为正交变换，用途完全不同"。

## 1. DCT-II 公式（正交归一化形式）

aurora 采用与 `scipy.fft.dct(..., norm='ortho')` 相同的**正交归一化**约定：

```
X[k] = s_k * sum_{n=0}^{N-1} x[n] * cos(pi * (2n+1) * k / (2N))

s_0 = sqrt(1/N)      <- k=0 的直流系数特殊缩放
s_k = sqrt(2/N)      <- k > 0 的所有系数
```

与"乘以 2 的无归一化版本"相比，正交归一化版本有三个优势：

- **矩阵是正交矩阵**：`M_ortho @ M_ortho.T = I`，逆变换就是转置（无需除以 2N）
- **能量守恒**（Parseval 定理）：`||X||^2 = ||x||^2`，压缩/重建时数值更稳定
- **与 scipy / numpy 生态兼容**：aurora 输出可直接与 librosa 的参考值比对

**为什么 k=0 单独处理？**

k=0 的基向量是常数 1，其 L2 模长为 `sqrt(N)`（而 k>0 的基向量模长为 `sqrt(N/2)`）。
为使所有基向量在同一尺度下正交，k=0 需额外除以 `sqrt(2)`，即 `s_0 = sqrt(1/N)`。

In [ ]:
# 演示：观察 DCT-II 的余弦基向量（N=8）
N = 8
k_vals = np.arange(N)
n_vals = np.arange(N)
# M[k, n] = cos(pi*(2n+1)*k / (2N))
M = np.cos(np.pi * np.outer(k_vals, 2*n_vals + 1) / (2*N))
print("余弦矩阵 M，shape:", M.shape)
print("第 0 行（k=0，常数基）:", np.round(M[0], 4))
print("第 1 行（k=1，最低频余弦）:", np.round(M[1], 4))
print("第 4 行（k=4，较高频余弦）:", np.round(M[4], 4))

## 2. DCT vs DFT：实值信号的更高效表示

DFT 假设信号以 N 为周期循环，当信号两端不连续时会产生"吉布斯效应（Gibbs phenomenon）"（频谱泄漏）。
DCT-II 等效于将信号做偶对称延拓后取 DFT 的实部，天然没有泄漏，输出为纯实数。

正交性验证：`M @ M.T` 的对角线为 N/2（k>0）或 N（k=0），非对角元为零，
说明余弦基向量两两正交（Cell 6 代码注释"应为 N/2=4 或 N=8"即来源于此）。
等价地，含归一化因子的 DCT 矩阵 D（scale[0]=√(1/N), scale[k>0]=√(2/N)）满足 D @ D.T = I，
正因如此，变换可精确逆转（逆矩阵等于转置）。

In [ ]:
# 演示：验证余弦矩阵的正交性
N = 8
k_vals = np.arange(N)
n_vals = np.arange(N)
M = np.cos(np.pi * np.outer(k_vals, 2*n_vals + 1) / (2*N))
G = M @ M.T  # Gram 矩阵，理论上对角线为 N/2（k>0）或 N（k=0）
print("M @ M.T 对角线（应为 N/2=4 或 N=8）:")
print(np.round(np.diag(G), 6))
print("非对角线最大绝对值（应≈0）:", np.max(np.abs(G - np.diag(np.diag(G)))))

## 3. 去相关性：从 Mel 能量到倒谱系数

Mel 滤波器组相邻通道重叠约 50%，导致相邻能量值的 Pearson 相关系数可达 0.9 以上。
DCT-II 将这批相关特征投影到正交余弦基上：低阶系数（k=0..12）捕获说话人/音素的整体频谱形状，
高阶系数（k>=13）捕获细节且相关性极低——这使得以欧氏距离为核心的分类器（GMM、DTW、神经网络）更加高效。
通常只取前 13 个系数，即 MFCC-13。

**用数字感受"去相关"的效果**（以真实语音帧为参考）：

| 特征对 | DCT 前（Mel 能量）| DCT 后（MFCC）|
|:---|:---:|:---:|
| 相邻系数 r(0,1) | ≈ 0.91 | ≈ 0.02 |
| 间隔 2 的系数 r(0,2) | ≈ 0.77 | ≈ 0.01 |
| 间隔 5 的系数 r(0,5) | ≈ 0.43 | < 0.01 |

Mel 空间里相邻 bin 因滤波器重叠而高度相关；DCT 后各系数近似正交，协方差矩阵接近对角阵。
这意味着 GMM、欧氏距离分类器等假设特征独立的模型可以直接在 MFCC 上有效工作，无需额外白化。

下面的演示用 AR(1) 过程模拟相关的 Mel 能量，直观展示 DCT 前后相关结构的变化：

In [ ]:
# 演示：模拟相关的 Mel 能量，经 DCT 后的系数分布
rng = np.random.default_rng(42)
# 用 AR(1) 过程模拟相邻 Mel bin 高度相关（phi=0.9）
N = 26
mel_energy = np.zeros(N)
mel_energy[0] = rng.standard_normal()
for i in range(1, N):
    mel_energy[i] = 0.9 * mel_energy[i-1] + 0.1 * rng.standard_normal()

# 手算 DCT
k = np.arange(N); n = np.arange(N)
M = np.cos(np.pi * np.outer(k, 2*n + 1) / (2*N))
dct_out = M @ mel_energy

print("Mel 能量（前5）:", np.round(mel_energy[:5], 4))
print("DCT 系数（全部，绝对值）:", np.round(np.abs(dct_out), 4))
print("能量集中度：前13系数占总能量的比例:",
      np.round(np.sum(dct_out[:13]**2) / np.sum(dct_out**2), 4))

## 4. ✏️ 实现 `dct_ii(x)`

**矩阵形式**（一次矩阵-向量乘法）：

```
D[k, n] = w(k) * cos(π·k·(2n+1) / (2N))
X = D @ x
```

**四步实现**：

| 步骤 | 代码 | 说明 |
|---|---|---|
| 1 | `N = len(x)` | 信号长度 |
| 2 | 构造 `D[k,n] = cos(π·k·(2n+1)/(2N))` | 用 `np.outer(k, 2n+1)` 广播 |
| 3 | 乘以归一化权重 `w(k)` | k=0: `1/√N`，k>0: `√(2/N)` |
| 4 | `return D @ x` | 矩阵乘法输出 `(N,)` |

**验收标准**：
- `np.allclose(dct_ii(x), scipy.fft.dct(x, norm='ortho'), atol=1e-10)` 通过
- 正交性：`D @ D.T ≈ I`（在 checker cell 中已验证）
- 纯 NumPy，不 import scipy

> AURORA 规则：实现算法，不用 black box。`scipy.fft.dct` 仅用于验证，不参与实现。

In [ ]:
def dct_ii(x: np.ndarray) -> np.ndarray:
    """DCT-II（正交归一化，norm='ortho'）：
    X[k] = s_k * sum_{n=0}^{N-1} x[n] * cos(pi*(2n+1)*k/(2N))
    s_0 = sqrt(1/N),  s_k = sqrt(2/N)  for k > 0
    """
    # ✏️ TODO: 计算 N, k = np.arange(N), n = np.arange(N)
    # ✏️ TODO: 构建余弦矩阵 M，shape (N, N)
    # ✏️ TODO: 计算 scale 向量（k=0 用 sqrt(1/N)，其余 sqrt(2/N)）
    # ✏️ TODO: return (M @ x) * scale
    raise NotImplementedError("TODO: 纯 NumPy 实现正交归一化 DCT-II：X[k] = s_k·Σ x[n]cos(π(2n+1)k/2N)")

In [ ]:
# 注意：参考实现只保留在 notebooks/5_audio_dsp/solutions/L49_dct_solutions.md

In [ ]:
# ✏️ 验收格：核对「你自己的」dct_ii（不做任何静默替换）
# 标准与第 4 节承诺一致：与 scipy.fft.dct(type=2, norm='ortho') 逐点 atol=1e-10。
# AURORA 规则：scipy 在这里只当"标准答案"对照，不参与实现。
from scipy.fft import dct as _scipy_dct

try:
    _rng_acc = np.random.default_rng(49)
    for _n in (4, 8, 26, 40):
        _x = _rng_acc.standard_normal(_n)
        _mine = np.asarray(dct_ii(_x))
        _ref = _scipy_dct(_x, type=2, norm="ortho")
        _err = float(np.max(np.abs(_mine - _ref)))
        assert _mine.shape == _ref.shape, f"N={_n}: 输出 shape {_mine.shape} != {_ref.shape}"
        assert np.allclose(_mine, _ref, atol=1e-10), (
            f"N={_n}: 你的 dct_ii 与 scipy.fft.dct(type=2, norm='ortho') "
            f"最大误差 {_err:.2e} > atol=1e-10，检查归一化权重 w(k) 和 cos 矩阵"
        )
        print(f"  N={_n:2d} ✓ 最大误差 {_err:.2e}")
    print("✅ 验收通过：你的 dct_ii 与 scipy.fft.dct(type=2, norm='ortho') 在 atol=1e-10 内逐点一致")
except (NotImplementedError, TypeError):
    print("⬜ dct_ii 还没实现——请先完成第 4 节的 TODO，再回来运行本验收格")

## 5. 参数实验：可逆性与截断重建

**实验 A — 可逆性验证**：正交归一化的 DCT-II 矩阵 $D$ 满足 $D D^T = I$，所以逆变换就是转置：

```
x_rec = D.T @ X    # 无需求逆矩阵，也无需 scipy
```

下方代码的 `_idct_ii_ref` 就是按这个式子写的纯 NumPy 参考逆变换。

**实验 B — 截断重建**：只保留前 `k_keep` 个 DCT 系数，其余置零后逆变换，观察重建误差随 `k_keep` 变化。
实验数据是一段 AR(1) 平滑相关向量（模拟相邻高度相关的 Mel 能量，幅度 std≈0.21）。
预期现象：`k_keep=3` 只剩大轮廓（RMSE≈0.09）；`k_keep=13` 时 RMSE≈0.03，相对信号幅度已很小——这正是 MFCC 截前 13 维的依据。
注意：能量集中只对"平滑相关"的向量成立；换成白噪声（`rng.standard_normal(26)`）试试，截掉一半系数就会丢掉约一半能量——DCT 压缩不了没有结构的数据。

In [ ]:
# 实验与练习解耦：若本地 dct_ii 还没写完，则临时用 aurora.audio.mfcc.dct_ii 跑后续实验。
from aurora.audio.mfcc import dct_ii as aurora_dct_ii

# numpy-based inverse DCT-II reference
def _idct_ii_ref(X):
    # Inverse of ortho DCT-II: X = D @ x  ⟹  x = D.T @ X
    # where D[k,n] = scale[k] * cos(pi*k*(2n+1)/(2N)),
    # scale[0]=sqrt(1/N), scale[k>0]=sqrt(2/N)
    N = len(X)
    k = np.arange(N)
    n = np.arange(N)
    scale = np.full(N, np.sqrt(2.0 / N))
    scale[0] = np.sqrt(1.0 / N)
    D = scale[:, None] * np.cos(np.pi * np.outer(k, 2 * n + 1) / (2 * N))
    return D.T @ X

# 用一段 26 维模拟 Mel 能量做实验：AR(1) 平滑相关（与第 3 节演示同款）
rng = np.random.default_rng(7)
N_mel = 26
x_mel = np.zeros(N_mel)
x_mel[0] = rng.standard_normal()
for i in range(1, N_mel):
    x_mel[i] = 0.9 * x_mel[i - 1] + 0.1 * rng.standard_normal()
print(f'模拟 Mel 能量：std={x_mel.std():.3f}（平滑相关，才有得压缩）')

try:
    probe = dct_ii(np.array([1.0, 2.0]))
except (NotImplementedError, TypeError):
    probe = None

_impl = dct_ii if probe is not None else aurora_dct_ii
if _impl is aurora_dct_ii:
    print('⚠️  dct_ii 尚未实现，使用 aurora.audio.mfcc.dct_ii 演示实验（完成 TODO 后可切换）')
X_dct = _impl(x_mel)

# 实验 A：精确逆变换
x_rec_full = _idct_ii_ref(X_dct)
print('实验 A — 精确重建误差（应≈0）:', np.max(np.abs(x_rec_full - x_mel)))

# 实验 B：截断重建
print('\n实验 B — 截断重建误差：')
for k_keep in [3, 6, 13]:
    X_trunc = X_dct.copy()
    X_trunc[k_keep:] = 0.0
    x_trunc_rec = _idct_ii_ref(X_trunc)
    rmse = np.sqrt(np.mean((x_trunc_rec - x_mel) ** 2))
    print(f'  k_keep={k_keep:2d} → RMSE={rmse:.4f}')

## 本课收束

`dct_ii(x)` 通过构建 N×N 余弦矩阵并做矩阵乘法，输出与输入等长的实值倒谱系数向量。
该函数是 `aurora.audio.mfcc` 模块中 MFCC 流水线的第三步，承接对数 Mel 能量谱，输出低阶系数即为 MFCC。
截断实验表明：对平滑相关的 Mel 型向量，前 13 个系数已能近似重建整条曲线（RMSE 只有信号幅度的一成左右），这解释了为何 MFCC-13 成为语音识别的标准特征维度。
下一课（L50）将把 `dct_ii` 嵌入完整的 MFCC 流水线：STFT → Mel 滤波 → 对数 → DCT → 截断，端到端处理真实音频帧。

## ✏️ 闭卷推导检查格 — DCT-II 正交归一化

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：正交归一化 DCT-II 的变换矩阵 $D$ 满足 $D D^T = I$。

其权重为：
$$w_k = \begin{cases} \sqrt{1/N} & k = 0 \\ \sqrt{2/N} & k > 0 \end{cases}$$

1. 写出矩阵元素 $D_{k,n}$ 的完整表达式（含权重 $w_k$）
2. 验证 $k=0$ 行与 $k=1$ 行正交（内积为 0）：手算 $\sum_{n=0}^{N-1} D_{0,n} D_{1,n}$

（在此处写推导...）

In [ ]:
# 验证：DCT 矩阵是否满足 D @ D.T ≈ I（正交性）
import numpy as np, sys
sys.path.insert(0, 'src')
from aurora.audio.mfcc import dct_ii

N = 16
x = np.eye(N)  # 输入单位矩阵 → 输出即 DCT 变换矩阵的转置
D = np.stack([dct_ii(x[n], norm='ortho') for n in range(N)])  # (N, N)

gram = D @ D.T
err = np.abs(gram - np.eye(N)).max()
assert err < 1e-10, f"D @ D.T 最大误差 {err:.2e}，不满足正交性"
print(f"✅ DCT-II 正交性验证通过（误差 = {err:.2e})")

In [ ]:
# ✏️ 本课自评
l49_review = {
    "dct_formula":             None,  # 记住 D[k,n]=w(k)·cos(πk(2n+1)/2N)，w(0)=1/√N？True/False
    "dct_ii_implemented":      None,  # dct_ii 实现并通过 scipy ortho 对比（atol<1e-10）？True/False
    "decorrelation_purpose":   None,  # 理解 DCT 去相关，把 Mel 相关能量旋转到倒谱坐标？True/False
    "mfcc13_reason":           None,  # 理解为什么取前 13 维（低频系数=音素信息）？True/False
    "whiteboard_passed":       None,  # 白板推导 DCT 正交归一化闭卷通过？True/False
}

unfilled = [k for k, v in l49_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l49_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L49 全部通关！进入 L50：MFCC 完整流水线')

---

→ **下一课**　[L50 · MFCC 完整流水线](L50_mfcc.ipynb)

> 下节课将学习 **MFCC 完整流水线**：信号 → STFT → Mel → log → DCT，每步输出形状确认。